# ❤️ Heart Disease / Stroke Classification Using ANN

## Project Objective
Build a binary classification model using an Artificial Neural Network (ANN) to predict the `Heart_ stroke` target from the supplied `heart_disease.csv` dataset.

### Workflow
Dataset → Data Inspection → Cleaning → EDA → Target Encoding → Train/Test Split → Preprocessing → Baseline ANN → Early Stopping → Dropout ANN → Evaluation → ROC-AUC → Save Model & Preprocessor → Test Prediction

**Important:** The final model and preprocessing pipeline are saved using the exact filenames expected by the Streamlit dashboard:
- `heart_disease_ann_model.keras`
- `heart_disease_preprocessor.pkl`

In [ ]:
# Import libraries
import os
import random
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Libraries imported successfully.")

In [ ]:
# Load dataset
DATA_FILE = "heart_disease.csv"

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f"{DATA_FILE} was not found. Put the CSV in the same folder as this notebook."
    )

df = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

In [ ]:
# Dataset inspection
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

In [ ]:
# First and last rows
display(df.head())
display(df.tail())

In [ ]:
# Missing values and duplicates
print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# Remove exact duplicate rows
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)

print(f"Rows before removing duplicates: {before}")
print(f"Rows after removing duplicates : {after}")
print(f"Duplicates removed             : {before - after}")

## Exploratory Data Analysis

The target column is `Heart_ stroke`. It contains two classes (`No` and `yes`), so this is a **binary classification** problem.

In [ ]:
# Target distribution
TARGET = "Heart_ stroke"

if TARGET not in df.columns:
    raise KeyError(f"Target column '{TARGET}' was not found.")

print(df[TARGET].value_counts(dropna=False))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET)
plt.title("Heart Stroke Target Distribution")
plt.xlabel("Heart_ stroke")
plt.ylabel("Count")
plt.show()

In [ ]:
# Numeric feature distributions
numeric_cols = df.select_dtypes(include=np.number).columns

df[numeric_cols].hist(figsize=(16, 12), bins=20)
plt.suptitle("Numeric Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numeric variables
plt.figure(figsize=(12, 9))
sns.heatmap(
    df[numeric_cols].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## Prepare Features and Target

The target is converted robustly to:
- `No` → `0`
- `Yes/yes` → `1`

The mapping is case-insensitive.

In [ ]:
# Separate features and target
X = df.drop(columns=[TARGET]).copy()

target_clean = df[TARGET].astype(str).str.strip().str.lower()

target_map = {
    "no": 0,
    "yes": 1
}

y = target_clean.map(target_map)

if y.isna().any():
    unexpected = sorted(target_clean[y.isna()].unique().tolist())
    raise ValueError(f"Unexpected target values found: {unexpected}")

y = y.astype(np.int32)

print("Target classes:")
print(y.value_counts().sort_index())

print("\nClass labels:")
print("0 = No")
print("1 = Yes")

print("\nX shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).sort_index())

## Preprocessing

The preprocessing pipeline is fitted **only on the training data** to avoid data leakage.

- Numerical features: median imputation + StandardScaler
- Categorical features: most-frequent imputation + OneHotEncoder

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_features = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

X_train_processed = np.asarray(X_train_processed, dtype=np.float32)
X_test_processed = np.asarray(X_test_processed, dtype=np.float32)

y_train_np = y_train.to_numpy(dtype=np.float32)
y_test_np = y_test.to_numpy(dtype=np.float32)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape :", X_test_processed.shape)

In [ ]:
# Processed feature names
processed_feature_names = preprocessor.get_feature_names_out()

print("Number of ANN input features:", len(processed_feature_names))

processed_preview = pd.DataFrame(
    X_train_processed[:5],
    columns=processed_feature_names
)

display(processed_preview)

## ANN Model

The project uses a binary-output ANN:
- ReLU hidden layers
- Sigmoid output layer
- Binary cross-entropy loss
- Adam optimizer

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

In [ ]:
# Baseline ANN
input_dim = X_train_processed.shape[1]

ann_baseline = Sequential([
    Input(shape=(input_dim,)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_baseline.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_baseline.summary()

In [ ]:
# Train baseline ANN
history_baseline = ann_baseline.fit(
    X_train_processed,
    y_train_np,
    epochs=100,
    batch_size=32,
    validation_split=0.20,
    verbose=1
)

In [ ]:
# Baseline training curves
baseline_history = pd.DataFrame(history_baseline.history)

plt.figure(figsize=(10, 5))
plt.plot(baseline_history["loss"], label="Training Loss")
plt.plot(baseline_history["val_loss"], label="Validation Loss")
plt.title("Baseline ANN - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(baseline_history["accuracy"], label="Training Accuracy")
plt.plot(baseline_history["val_accuracy"], label="Validation Accuracy")
plt.title("Baseline ANN - Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## ANN with Early Stopping

Early stopping stops training when validation loss stops improving and restores the best weights.

In [ ]:
ann_early = Sequential([
    Input(shape=(input_dim,)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_early.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop_early = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    restore_best_weights=True
)

history_early = ann_early.fit(
    X_train_processed,
    y_train_np,
    epochs=150,
    batch_size=32,
    validation_split=0.20,
    callbacks=[early_stop_early],
    verbose=1
)

print("Training stopped after", len(history_early.history["loss"]), "epochs.")

In [ ]:
# Early stopping curves
early_history = pd.DataFrame(history_early.history)

plt.figure(figsize=(10, 5))
plt.plot(early_history["loss"], label="Training Loss")
plt.plot(early_history["val_loss"], label="Validation Loss")
plt.title("ANN with Early Stopping - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(early_history["accuracy"], label="Training Accuracy")
plt.plot(early_history["val_accuracy"], label="Validation Accuracy")
plt.title("ANN with Early Stopping - Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## Final ANN with Dropout

Dropout is added to reduce overfitting.

In [ ]:
final_model = Sequential([
    Input(shape=(input_dim,)),
    Dense(32, activation="relu"),
    Dropout(0.30),
    Dense(16, activation="relu"),
    Dropout(0.20),
    Dense(1, activation="sigmoid")
])

final_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

final_model.summary()

In [ ]:
# Use a fresh callback for the final model
early_stop_final = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    restore_best_weights=True
)

history_final = final_model.fit(
    X_train_processed,
    y_train_np,
    epochs=150,
    batch_size=32,
    validation_split=0.20,
    callbacks=[early_stop_final],
    verbose=1
)

print("Final training epochs:", len(history_final.history["loss"]))

In [ ]:
# Final ANN training curves
final_history = pd.DataFrame(history_final.history)

plt.figure(figsize=(10, 5))
plt.plot(final_history["loss"], label="Training Loss")
plt.plot(final_history["val_loss"], label="Validation Loss")
plt.title("Final ANN - Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(final_history["accuracy"], label="Training Accuracy")
plt.plot(final_history["val_accuracy"], label="Validation Accuracy")
plt.title("Final ANN - Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## Model Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

# Test-set evaluation
test_loss, test_accuracy = final_model.evaluate(
    X_test_processed,
    y_test_np,
    verbose=0
)

y_prob = final_model.predict(
    X_test_processed,
    verbose=0
).ravel()

y_pred = (y_prob >= 0.5).astype(int)

accuracy = accuracy_score(y_test_np, y_pred)
precision = precision_score(y_test_np, y_pred, zero_division=0)
recall = recall_score(y_test_np, y_pred, zero_division=0)
f1 = f1_score(y_test_np, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test_np, y_prob)

metrics_df = pd.DataFrame({
    "Metric": [
        "Test Loss",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Value": [
        test_loss,
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]
})

display(metrics_df)

print(classification_report(
    y_test_np,
    y_pred,
    target_names=["No", "Yes"],
    zero_division=0
))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test_np, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted No", "Predicted Yes"],
    yticklabels=["Actual No", "Actual Yes"]
)
plt.title("Confusion Matrix - Final ANN")
plt.xlabel("Prediction")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test_np, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ANN (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Final ANN")
plt.legend()
plt.tight_layout()
plt.show()

## Save Model and Preprocessor

These filenames are intentionally matched to the Streamlit dashboard.

In [ ]:
# Save the final ANN model
MODEL_FILE = "heart_disease_ann_model.keras"
PREPROCESSOR_FILE = "heart_disease_preprocessor.pkl"

final_model.save(MODEL_FILE)

with open(PREPROCESSOR_FILE, "wb") as f:
    pickle.dump(preprocessor, f)

print("Saved successfully:")
print("-", os.path.abspath(MODEL_FILE))
print("-", os.path.abspath(PREPROCESSOR_FILE))

assert os.path.exists(MODEL_FILE), "Model file was not created."
assert os.path.exists(PREPROCESSOR_FILE), "Preprocessor file was not created."

print("\nFile verification: PASSED")

## Verify Saved Files

Run this cell before opening the Streamlit dashboard.

In [ ]:
print("Required dashboard files:")

for filename in [
    "heart_disease.csv",
    MODEL_FILE,
    PREPROCESSOR_FILE
]:
    print(f"{filename}: {'FOUND' if os.path.exists(filename) else 'MISSING'}")

## Example Prediction

The example below automatically uses the actual columns in the dataset and therefore avoids hard-coding a possibly incorrect feature list.

In [ ]:
# Create one example patient from the dataset's typical values
sample_input = {}

for col in X.columns:
    if col in numeric_features:
        sample_input[col] = X_train[col].median()
    else:
        mode_values = X_train[col].mode(dropna=True)
        sample_input[col] = mode_values.iloc[0] if not mode_values.empty else X_train[col].iloc[0]

sample_patient = pd.DataFrame([sample_input])

display(sample_patient)

In [ ]:
# Transform and predict the example patient
sample_processed = preprocessor.transform(sample_patient)
sample_processed = np.asarray(sample_processed, dtype=np.float32)

sample_probability = float(
    final_model.predict(sample_processed, verbose=0).ravel()[0]
)

sample_prediction = int(sample_probability >= 0.5)

print(f"Predicted probability of Yes: {sample_probability:.4f}")
print(
    "Prediction:",
    "Yes - Heart Disease/Stroke" if sample_prediction == 1
    else "No - No Heart Disease/Stroke"
)

## Final Project Files

After running all cells successfully, the folder should contain:

```text
heart_disease_ann_model.ipynb
heart_disease.csv
heart_disease_ann_model.keras
heart_disease_preprocessor.pkl
app.py
requirements.txt
```

Then run the dashboard from the same folder:

```powershell
python -m streamlit run app.py
```